# CW2 Scaffold A: Blockchain Fraud Detection

Rare-event classification on the Elliptic Bitcoin dataset.

**Module:** FIN306 / FIN510  
**Instructions:** Run provided cells to produce baseline results. Complete the `# TODO` sections. Write a 2,500-word reflective report addressing the five components in the CW2 brief.

**Data:** Elliptic Bitcoin labelled subset — loaded automatically from `fin510-colab-notebooks/resources/elliptic/` on Colab, or from `data/elliptic/` if you have cloned the course repo locally. No manual download required.

---

## What you will do

| Step | Task | Status |
|------|------|--------|
| 1 | Load Elliptic Bitcoin labelled data | Provided |
| 2 | Explore temporal drift in illicit transaction rate | **TODO 1** |
| 3 | Baseline: shuffled cross-validation | Provided |
| 4 | Walk-forward temporal validation | Provided |
| 5 | Visualise look-ahead bias gap | **TODO 2** |
| 6 | Cost-sensitive threshold optimisation | Provided |
| 7 | Compare cost curves across regulatory scenarios | **TODO 3** |
| 8 | Model comparison: logistic, random forest, gradient boosting | Provided |
| 9 | Visualise model performance across time windows | **TODO 4** |
| 10 | Extension: add graph features and re-evaluate | **TODO 5 (optional)** |

---

## Key concepts

**Rare-event classification** poses unique challenges: class imbalance (typically <5% fraud), high cost asymmetry (false negatives cost far more than false positives), and evolving fraud patterns over time. Standard accuracy is meaningless when 98% of transactions are legitimate.

**Temporal validation** is essential for fraud detection. Fraudsters adapt to detection systems, so models trained on historical patterns degrade over time. Walk-forward validation simulates real deployment: train on past data, test on future data, never allow future information to leak into training.

**Look-ahead bias** occurs when evaluation methods (like shuffled cross-validation) randomly split data, allowing the model to learn from "future" fraud patterns. This inflates performance metrics and creates false confidence in model robustness.

**Cost-sensitive classification** optimises decision thresholds based on the relative costs of false positives (investigating legitimate transactions) and false negatives (missing fraud). The optimal threshold depends on regulatory requirements, customer experience constraints, and business model.

**AUC (Area Under ROC Curve)** measures classifier discrimination ability across all possible thresholds. Average Precision (AP) is more appropriate for imbalanced datasets, focusing on the high-recall region where fraud detection operates.

**Model complexity and temporal stability**: More complex models (random forests, gradient boosting) may fit historical patterns better but degrade faster when fraud typologies evolve. This is the deployment risk : high training performance, poor future performance.

## Before you code: state your research claim

Good data science starts with a question that evidence can refute. Before running any cells, read the claim below and ask yourself the four questions that follow.

---

**Starting claim:**

> *Standard shuffled cross-validation overstates fraud detection AUC on the Elliptic Bitcoin dataset relative to temporal walk-forward validation, because fraud typologies evolve over time and shuffled CV allows future patterns to leak into training.*

---

**Think through these questions before you start:**

1. **Why would you expect this to be true?** What is it about fraud — as opposed to, say, credit scoring on cross-sectional data — that makes temporal ordering especially important?

2. **What would falsify it?** If shuffled AUC and walk-forward AUC are nearly identical on this dataset, what would that tell you about the nature of temporal drift in the Elliptic data?

3. **Is statistical significance enough?** Suppose you find a gap of 0.065 AUC. Is that economically meaningful? Think about what a 6.5 percentage-point difference in AUC means when a bank screens 50,000 transactions per day.

4. **What alternative explanations exist?** Could the gap be driven by class imbalance changing over time, rather than fraud pattern evolution? How would you distinguish between these?

---

Write a one- or two-sentence version of *your own* refined claim in the markdown cell below — before running any code. Your report introduction should open with this claim.

**Your refined claim (write here before running any code):**

*Replace this text with your own one- or two-sentence falsifiable claim. You will return to this cell after completing the analysis and assess whether your results support or refute it.*

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, precision_recall_curve,
)
from pathlib import Path

pd.set_option("display.float_format", "{:.4f}".format)
print("Setup complete.")


## Step 1: Load Elliptic (labelled subset)

In [ ]:
ELL_URL = (
    "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
    "main/resources/elliptic/elliptic_labelled_v2.parquet"
)
# Try common local paths (repo clone, lab working dir) before falling back to URL.
ell_path = None
for _p in [
    Path("data/elliptic/elliptic_labelled.parquet"),
    Path("../data/elliptic/elliptic_labelled.parquet"),
    Path("resources/elliptic/elliptic_labelled.parquet"),
    Path("elliptic_labelled.parquet"),
]:
    if _p.is_file():
        ell_path = _p
        break
if ell_path is None:
    ell_path = Path("elliptic_labelled.parquet")
    import urllib.request
    print("Downloading Elliptic labelled parquet from GitHub (~25 MB)...")
    urllib.request.urlretrieve(ELL_URL, ell_path)
    print("Done.")

print(f"Loaded from: {ell_path}")
ell = pd.read_parquet(ell_path)
feat_cols = [c for c in ell.columns if c.startswith("feat_")]
X = ell[feat_cols].values
y = (ell["class"] == 1).astype(int).values
ts = ell["time_step"].values

print(f"Transactions:  {len(ell):,}")
print(f"Features:      {len(feat_cols)}")
print(f"Illicit:       {y.sum():,} ({y.mean():.1%})")
print(f"Time steps:    {ts.min()} to {ts.max()}")


### TODO 1: Plot illicit rate by `time_step` (temporal drift)

**Task:** Create a line plot showing the illicit transaction rate (% of transactions that are fraud) for each time step from 1 to 49.

**Why this matters:** If the illicit rate changes over time, shuffled cross-validation will mix high-fraud and low-fraud periods in both training and test sets, creating look-ahead bias. Temporal validation respects the time ordering.

**Code guidance:**
```python
# Group by time_step, calculate illicit rate
illicit_by_time = ell.groupby('time_step')['class'].apply(lambda x: (x == 1).mean())

# Plot
plt.figure(figsize=(10, 4))
plt.plot(illicit_by_time.index, illicit_by_time.values, marker='o', linewidth=1)
plt.xlabel('Time step')
plt.ylabel('Illicit transaction rate')
plt.title('Temporal drift in fraud patterns (Elliptic Bitcoin)')
plt.grid(alpha=0.3)
plt.show()
```

**For your report (Section B: Data Quality):** 
- Does the illicit rate trend upward, downward, or remain stable? 
- What does this tell you about the data generating process?
- How does temporal drift affect model deployment risk?

In [ ]:
# TODO 1: your plot here


## Step 2: Baseline (shuffled stratified CV)

In [ ]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_shuffled = cross_val_score(pipe_lr, X, y, cv=cv, scoring="roc_auc")
ap_shuffled = cross_val_score(pipe_lr, X, y, cv=cv, scoring="average_precision")
print("Shuffled 5-fold CV (logistic, balanced)")
print(f"  AUC: {auc_shuffled.mean():.3f} +/- {auc_shuffled.std():.3f}")
print(f"  AP:  {ap_shuffled.mean():.3f} +/- {ap_shuffled.std():.3f}")


## Step 3: Walk-forward temporal validation

In [ ]:
boundaries = [1, 10, 20, 30, 40, 50]
wf_results = []
for i in range(len(boundaries) - 2):
    train_mask = (ts >= boundaries[0]) & (ts < boundaries[i + 1])
    test_mask = (ts >= boundaries[i + 1]) & (ts < boundaries[i + 2])
    if train_mask.sum() < 50 or test_mask.sum() < 50:
        continue
    X_tr, y_tr = X[train_mask], y[train_mask]
    X_te, y_te = X[test_mask], y[test_mask]
    pipe_lr.fit(X_tr, y_tr)
    auc_wf = roc_auc_score(y_te, pipe_lr.predict_proba(X_te)[:, 1])
    wf_results.append({
        "window": f"t={boundaries[i+1]}-{boundaries[i+2]-1}",
        "AUC": auc_wf,
        "illicit_rate": y_te.mean(),
    })
    print(f"  {wf_results[-1]['window']}: AUC={auc_wf:.3f} illicit={y_te.mean():.1%}")

mean_wf = np.mean([r["AUC"] for r in wf_results])
print(f"\nWalk-forward AUC (mean): {mean_wf:.3f}")
print(f"Shuffled CV AUC:         {auc_shuffled.mean():.3f}")
print(f"Look-ahead bias gap:     {auc_shuffled.mean() - mean_wf:+.3f}")


### TODO 2: Dual-axis plot of walk-forward AUC and illicit rate per window

**Task:** Create a plot with two y-axes:
- Left axis: Walk-forward AUC for each test window (bars)
- Right axis: Illicit rate for each test window (line)
- Add a horizontal line showing the shuffled CV AUC (0.97 from Step 2)

**Why this matters:** This visualises the look-ahead bias gap and shows whether AUC degradation correlates with changes in illicit rate (data drift) or reflects model overfitting to historical patterns.

**Code guidance:**
```python
fig, ax1 = plt.subplots(figsize=(10, 5))

windows = [r['window'] for r in wf_results]
aucs = [r['AUC'] for r in wf_results]
rates = [r['illicit_rate'] for r in wf_results]

# Bar plot: AUC
ax1.bar(windows, aucs, alpha=0.7, color='steelblue', label='Walk-forward AUC')
ax1.axhline(auc_shuffled.mean(), color='red', linestyle='--', label='Shuffled CV AUC')
ax1.set_xlabel('Test window')
ax1.set_ylabel('AUC', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_ylim(0.7, 1.0)
ax1.legend(loc='upper left')

# Line plot: illicit rate
ax2 = ax1.twinx()
ax2.plot(windows, rates, color='darkred', marker='o', linewidth=2, label='Illicit rate')
ax2.set_ylabel('Illicit rate', color='darkred')
ax2.tick_params(axis='y', labelcolor='darkred')
ax2.legend(loc='upper right')

plt.title('Walk-forward AUC vs. shuffled CV: temporal validation exposes look-ahead bias')
plt.tight_layout()
plt.show()
```

**For your report (Section C: Results Interpretation):**
- How large is the look-ahead bias gap (shuffled AUC - mean walk-forward AUC)?
- Does AUC degrade as illicit rate changes, suggesting data drift?
- What does this mean for real-world deployment?

In [ ]:
# TODO 2: your figure here


## Step 4: Cost-sensitive threshold (last WF test window)

In [ ]:
last_train = (ts >= 1) & (ts < 40)
last_test = (ts >= 40) & (ts < 50)
pipe_lr.fit(X[last_train], y[last_train])
y_prob = pipe_lr.predict_proba(X[last_test])[:, 1]
y_te = y[last_test]

cost_fp, cost_fn = 50, 5000
thresholds = np.arange(0.01, 0.95, 0.01)
costs = []
for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_te, y_pred)
    tn, fp, fn, tp = cm.ravel()
    costs.append((fp * cost_fp + fn * cost_fn) / len(y_te))
opt_t = thresholds[int(np.argmin(costs))]
print(f"Optimal threshold (base costs): {opt_t:.2f}")


### TODO 3: Repeat cost curves for cost_fn in {1000, 5000, 50000}; plot all three

**Task:** The base case uses `cost_fp=50` (investigating a legitimate transaction) and `cost_fn=5000` (missing fraud). Regulatory pressure or business model changes could shift these costs. Generate cost curves for three scenarios:
1. Low penalty: `cost_fn=1000` (lenient regime, high customer friction tolerance)
2. Base case: `cost_fn=5000` (current assumptions)
3. High penalty: `cost_fn=50000` (strict AML enforcement, reputational risk)

Plot all three curves on the same axes, marking optimal thresholds.

**Why this matters:** Fraud detection thresholds are policy decisions, not purely statistical. Regulators impose costs for missed fraud (fines, sanctions); customers impose costs for false positives (account freezes, delays). Your model must adapt to the regulatory environment.

**Code guidance:**
```python
fig, ax = plt.subplots(figsize=(10, 5))

for cost_fn, color, label in [(1000, 'green', 'Low penalty (fn=1000)'),
                                (5000, 'orange', 'Base case (fn=5000)'),
                                (50000, 'red', 'High penalty (fn=50000)')]:
    costs = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        cm = confusion_matrix(y_te, y_pred)
        tn, fp, fn, tp = cm.ravel()
        costs.append((fp * cost_fp + fn * cost_fn) / len(y_te))
    opt_t = thresholds[int(np.argmin(costs))]
    ax.plot(thresholds, costs, color=color, linewidth=2, label=f'{label}, opt={opt_t:.2f}')
    ax.axvline(opt_t, color=color, linestyle='--', alpha=0.5)

ax.set_xlabel('Classification threshold')
ax.set_ylabel('Expected cost per transaction')
ax.set_title('Cost-sensitive threshold selection under different regulatory scenarios')
ax.legend()
ax.grid(alpha=0.3)
plt.show()
```

**For your report (Section E: FinTech Applications and Ethics):**
- How does the optimal threshold change as `cost_fn` increases?
- What does a higher threshold mean for customers (more/fewer investigations)?
- How would UK FCA AML guidance or PSD2 SCA requirements influence these trade-offs?
- What ethical concerns arise when optimising thresholds purely for cost minimisation?

In [ ]:
# TODO 3: your plots here


## Step 5: Model comparison (walk-forward)

In [ ]:
models = {
    "Logistic": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("clf", RandomForestClassifier(
            n_estimators=200, max_depth=10, class_weight="balanced",
            random_state=42, n_jobs=-1)),
    ]),
    "Gradient Boosting": Pipeline([
        ("clf", GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)),
    ]),
}
for name, pipe in models.items():
    aucs = []
    for i in range(len(boundaries) - 2):
        tr = (ts >= 1) & (ts < boundaries[i + 1])
        te = (ts >= boundaries[i + 1]) & (ts < boundaries[i + 2])
        if tr.sum() < 50 or te.sum() < 50:
            continue
        pipe.fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], pipe.predict_proba(X[te])[:, 1]))
    print(f"{name:18s} mean WF AUC = {np.mean(aucs):.3f}")


### TODO 4: Grouped bar chart by model and window; interpret bias-variance and deployment risk

**Task:** Create a grouped bar chart showing walk-forward AUC for each model (Logistic, Random Forest, Gradient Boosting) across all test windows. This visualises which models degrade fastest as fraud patterns evolve.

**Why this matters:** Complex models (Random Forest, Gradient Boosting) often achieve higher AUC on training data, but their performance may degrade faster on future data if they overfit to historical fraud typologies. Simpler models (Logistic Regression) have higher bias but may be more stable over time. This is the bias-variance-stability tradeoff for deployed fraud detection.

**Code guidance:**
```python
import matplotlib.pyplot as plt
import numpy as np

# Re-run walk-forward for all models, store results
all_results = {name: [] for name in models.keys()}
for name, pipe in models.items():
    for i in range(len(boundaries) - 2):
        tr = (ts >= 1) & (ts < boundaries[i + 1])
        te = (ts >= boundaries[i + 1]) & (ts < boundaries[i + 2])
        if tr.sum() < 50 or te.sum() < 50:
            continue
        pipe.fit(X[tr], y[tr])
        auc = roc_auc_score(y[te], pipe.predict_proba(X[te])[:, 1])
        all_results[name].append(auc)

# Plot grouped bars
windows = [f"t={boundaries[i+1]}-{boundaries[i+2]-1}" for i in range(len(boundaries)-2) 
           if (ts >= 1) & (ts < boundaries[i+1]).sum() >= 50 
           and (ts >= boundaries[i+1]) & (ts < boundaries[i+2]).sum() >= 50]

x = np.arange(len(windows))
width = 0.25
fig, ax = plt.subplots(figsize=(12, 5))

for i, (name, aucs) in enumerate(all_results.items()):
    ax.bar(x + i*width, aucs, width, label=name)

ax.set_xlabel('Test window')
ax.set_ylabel('AUC')
ax.set_title('Model comparison: walk-forward AUC across time windows')
ax.set_xticks(x + width)
ax.set_xticklabels(windows)
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Compute stability (std dev across windows)
for name, aucs in all_results.items():
    print(f"{name:18s} | Mean: {np.mean(aucs):.3f} | Std: {np.std(aucs):.3f}")
```

**For your report (Section C: Results Interpretation and Section D: Limitations):**
- Which model has the highest mean AUC? Is the difference economically significant?
- Which model has the lowest standard deviation (most stable over time)?
- If Random Forest has higher mean AUC but also higher variance across windows, would you deploy it? Why or why not?
- What does temporal instability tell you about overfitting to historical fraud patterns?
- How would you decide between a complex model (high mean, high variance) and a simple model (lower mean, lower variance) for real-world deployment?

In [ ]:
# TODO 4: your chart here


## Step 6 (extension): graph degrees + re-fit

In [ ]:
import networkx as nx

EDGES_URL = (
    "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
    "main/resources/elliptic/elliptic_edges_labelled_v2.parquet"
)
edges_path = None
for _p in [
    Path("data/elliptic/elliptic_edges_labelled.parquet"),
    Path("../data/elliptic/elliptic_edges_labelled.parquet"),
    Path("resources/elliptic/elliptic_edges_labelled.parquet"),
    Path("elliptic_edges_labelled.parquet"),
]:
    if _p.is_file():
        edges_path = _p
        break
if edges_path is None:
    edges_path = Path("elliptic_edges_labelled.parquet")
    import urllib.request
    print("Downloading Elliptic edges parquet from GitHub...")
    urllib.request.urlretrieve(EDGES_URL, edges_path)
edges_df = pd.read_parquet(edges_path)
G = nx.from_pandas_edgelist(
    edges_df, source="txId1", target="txId2", create_using=nx.DiGraph
)
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())
ell = ell.copy()
ell["in_degree"] = ell["txId"].map(in_deg).fillna(0).astype(int)
ell["out_degree"] = ell["txId"].map(out_deg).fillna(0).astype(int)
ell["total_degree"] = ell["in_degree"] + ell["out_degree"]
print("Graph features merged (extension).")


### TODO 5 (Extension): Stack degree features onto X; re-run walk-forward for your best model

**Task:** The Elliptic dataset includes a transaction graph (who paid whom). Graph structure features — in-degree (how many incoming payments), out-degree (how many outgoing payments), total degree — may signal fraud patterns (e.g., money laundering often creates hub-and-spoke structures).

Add the three degree features (`in_degree`, `out_degree`, `total_degree`) to your feature matrix and re-run walk-forward validation for your best-performing model from Step 5. Compare performance with and without graph features.

**Why this matters:** Real fraud detection systems use multiple data sources (transaction features, graph structure, device fingerprints, behavioural patterns). This extension tests whether network topology improves predictive power beyond transaction-level features.

**Code guidance:**
```python
# Stack degree features onto original features
X_with_graph = np.hstack([
    X,  # original 166 features
    ell[['in_degree', 'out_degree', 'total_degree']].values  # 3 new features
])

# Re-run walk-forward for your best model (e.g., Gradient Boosting)
best_model_name = "Gradient Boosting"  # Replace with your choice
best_pipe = models[best_model_name]

aucs_original = []
aucs_with_graph = []

for i in range(len(boundaries) - 2):
    tr = (ts >= 1) & (ts < boundaries[i + 1])
    te = (ts >= boundaries[i + 1]) & (ts < boundaries[i + 2])
    if tr.sum() < 50 or te.sum() < 50:
        continue
    
    # Original features
    best_pipe.fit(X[tr], y[tr])
    aucs_original.append(roc_auc_score(y[te], best_pipe.predict_proba(X[te])[:, 1]))
    
    # With graph features
    best_pipe.fit(X_with_graph[tr], y[tr])
    aucs_with_graph.append(roc_auc_score(y[te], best_pipe.predict_proba(X_with_graph[te])[:, 1]))

print(f"{best_model_name} without graph features: {np.mean(aucs_original):.3f}")
print(f"{best_model_name} with graph features:    {np.mean(aucs_with_graph):.3f}")
print(f"Improvement:                           {np.mean(aucs_with_graph) - np.mean(aucs_original):+.3f}")
```

**For your report (Section A: Method Rationale or Section C: Results):**
- Do graph features improve AUC? By how much?
- Is the improvement economically meaningful for a deployed system?
- What trade-offs exist when adding graph features (data availability, computational cost, privacy concerns)?
- Would you recommend including graph features in production? Why or why not?

**Note:** This TODO is optional but demonstrates deeper engagement with the dataset and real-world fraud detection methods. Including it strengthens Section A (method understanding) and Section C (thoroughness of analysis).

## Report Guidance: Connecting Your Analysis to the Five CW2 Components

Now that you have completed the analysis, write your 2,500-word reflective report addressing the five components from the CW2 brief. Use your results from this notebook as evidence, but focus on **interpretation, critical evaluation, and limitations** rather than just describing what you did.

---

### A. Method Rationale and Theoretical Foundation (500 words)

**Key questions to address:**
- Why is fraud detection a rare-event classification problem, and why does that matter?
- Why is temporal validation essential for fraud detection (vs. shuffled CV)?
- What is look-ahead bias, and why does it inflate performance metrics?
- Why use AUC and Average Precision rather than accuracy for imbalanced classification?
- What theoretical framework underpins cost-sensitive classification?
- How does this connect to module concepts (bias-variance tradeoff, overfitting, deployment risk)?

**Evidence from your analysis:**
- Your temporal drift plot (TODO 1) shows fraud patterns evolve over time
- Your look-ahead bias gap (TODO 2) quantifies the risk of using shuffled CV
- Your cost curves (TODO 3) show how optimal thresholds depend on regulatory context

---

### B. Data Quality and Preparation Decisions (500 words)

**Key questions to address:**
- What data quality concerns exist in the Elliptic dataset? (Labelling accuracy, class imbalance, temporal drift, survivorship bias in the labelled subset)
- How did you handle class imbalance? (Balanced class weights, stratified splitting, threshold optimisation)
- What preprocessing choices did you make? (Feature scaling for logistic regression, no scaling for tree-based models)
- How might data quality issues affect your results? (Mislabelled transactions, evolving fraud typologies, limited time coverage)

**Connection to CW1:**
Think of this section as a narrative version of your CW1 risk register, applied to the Elliptic dataset. Identify the 2-3 most significant data quality risks (e.g., temporal drift, labelling error, class imbalance), explain how they arise, and describe how you mitigated or disclosed them.

**Evidence from your analysis:**
- Your temporal drift plot (TODO 1) shows how illicit rate changes over time
- The walk-forward validation design mitigates look-ahead bias
- Class imbalance (~2% fraud) requires balanced class weights and cost-sensitive thresholds

---

### C. Results Interpretation and Validation (750 words)

**Key questions to address:**
- What do your results show? (Shuffled CV AUC ~0.97, walk-forward AUC ~0.85, look-ahead bias gap ~0.12)
- Is the look-ahead bias gap economically meaningful? (For a bank screening 50,000 transactions/day, a 12-point AUC gap could mean thousands more false positives or missed fraud cases)
- How do the three models (Logistic, Random Forest, Gradient Boosting) compare in mean AUC and temporal stability?
- Which model would you deploy, and why? (Trade-off between mean performance and variance across windows)
- Do graph features improve performance? (TODO 5 extension - compare AUC with/without degree features)
- How robust are your findings? (Are results consistent across different time windows, or do they degrade sharply in later periods?)

**Evidence from your analysis:**
- Your dual-axis plot (TODO 2) shows look-ahead bias and correlation with illicit rate drift
- Your model comparison chart (TODO 4) shows which models are most stable over time
- Mean AUC and standard deviation across windows quantify the bias-variance-stability tradeoff

---

### D. Limitations, Assumptions, and Potential Pitfalls (500 words)

**Key questions to address:**
- What are the key limitations of this analysis?
  - Limited temporal coverage (49 time steps from Elliptic)
  - Labelled subset may not be representative of all Bitcoin transactions
  - Binary classification (licit/illicit) oversimplifies real fraud typologies
  - No transaction costs, computational constraints, or latency requirements modelled
- Which assumptions are most critical (and most questionable)?
  - Fraud typologies evolve smoothly over time (not sudden regime shifts)
  - Labelled transactions are representative (no systematic mislabelling)
  - Cost assumptions (cost_fp=50, cost_fn=5000) reflect real regulatory/business constraints
- What biases might exist?
  - Look-ahead bias (if using shuffled CV in production)
  - Selection bias (labelled subset may over-represent caught fraud)
  - Data snooping (if threshold optimised on test set)
- What would you do differently with more time/resources?
  - Longer time series to test temporal stability further out-of-sample
  - Incorporate real transaction costs, latency constraints, explainability requirements
  - Test on multiple cryptocurrencies (Bitcoin, Ethereum, etc.)

---

### E. Appropriate Use in FinTech Contexts and Ethics (250 words)

**Key questions to address:**
- How could this method be applied in a FinTech setting?
  - Payment processors (PayPal, Stripe) screening transactions in real-time
  - Cryptocurrency exchanges (Coinbase, Binance) detecting money laundering
  - Digital banks (Revolut, Monzo) complying with AML/CTF regulations
- What ethical considerations arise?
  - **Fairness**: Do fraud models disproportionately flag certain customer demographics?
  - **Transparency**: Can you explain to a customer why their transaction was flagged?
  - **Accountability**: Who is responsible when the model misses fraud or freezes a legitimate account?
- What regulatory constraints apply?
  - UK FCA AML guidance (risk-based approach, ongoing monitoring)
  - GDPR (right to explanation for automated decisions)
  - PSD2 Strong Customer Authentication (balance fraud prevention with customer friction)
- What professional standards should practitioners follow?
  - Disclose model limitations and uncertainty to decision-makers
  - Monitor for temporal drift and retrain regularly
  - Implement human review for high-stakes decisions (large transactions, account freezes)

**Evidence from your analysis:**
- Your cost curves (TODO 3) show how regulatory penalties shape optimal thresholds
- Temporal instability (TODO 4) highlights the need for ongoing model monitoring
- Look-ahead bias (TODO 2) demonstrates why rigorous validation is essential before deployment

In [ ]:
# TODO 5: your extension here
